# LCOE Calculator: Scenario Analysis for Communities

This notebook demonstrates the LCOE (Levelized Cost of Electricity) calculator for different communities and technology scenarios.

**Scenario**: Evaluate PV system viability across 3 priority zones (Cacula, Humpata, Quilengues) with varying capacity and irradiance assumptions.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add scripts to path
sys.path.insert(0, str(Path.cwd() / 'scripts'))

from lcoe_calculator import LCOECalculator, SolarParameters

# Load communities data
communities_path = Path('data/processed/communities_45.csv')
communities = pd.read_csv(communities_path)

# Filter to priority zones
priority_zones = communities[communities['name'].isin(['Cacula', 'Humpata', 'Quilengues'])].copy()
print(f"Priority Zones:\n{priority_zones}\n")

## 1. Single Zone Analysis: Cacula (Zona A)

Calculate LCOE for different system sizes and assumptions.

In [ ]:
# Zone A: Cacula parameters
zone_a = priority_zones[priority_zones['name'] == 'Cacula'].iloc[0]
capacity_mw_a = 1.0  # 1 MW pilot
annual_irradiance_a = 6.1 * 365  # kWh/m²/year based on maps

# Initialize calculator
calc = LCOECalculator(location='Huíla')

# Compare technologies
comparison_a = calc.compare_technologies(
    capacity_mw=capacity_mw_a,
    annual_irradiance=annual_irradiance_a,
    discount_rate=8,
    lifetime=25
)

print(f"Zone A (Cacula) - {capacity_mw_a} MW, {annual_irradiance_a:.0f} kWh/m²/year")
print(comparison_a[['technology_name', 'lcoe_usd_per_kwh', 'capex_per_kw', 'annual_generation_mwh']])
print()

# Best option
best_a = comparison_a.sort_values('lcoe_usd_per_kwh').iloc[0]
print(f"RECOMMENDED: {best_a['technology_name']} @ ${best_a['lcoe_usd_per_kwh']:.3f}/kWh")

## 2. Multi-Zone Comparison

Compare LCOE across all 3 priority zones with same capacity (1 MW).

In [ ]:
# Irradiance by zone (from maps)
irradiance_map = {
    'Cacula': 6.1 * 365,      # kWh/m²/year
    'Humpata': 6.3 * 365,
    'Quilengues': 5.9 * 365
}

lcoe_summary = []

for idx, zone in priority_zones.iterrows():
    zone_name = zone['name']
    annual_irr = irradiance_map[zone_name]
    
    result = calc.compare_technologies(
        capacity_mw=1.0,
        annual_irradiance=annual_irr,
        discount_rate=8,
        lifetime=25
    )
    
    best = result.sort_values('lcoe_usd_per_kwh').iloc[0]
    lcoe_summary.append({
        'Zone': zone_name,
        'Population': zone['population_est'],
        'Annual_Irradiance_kWh_m2': annual_irr,
        'Best_Technology': best['technology_name'],
        'LCOE_USD_per_kWh': best['lcoe_usd_per_kwh'],
        'Annual_Generation_MWh': best['annual_generation_mwh']
    })

summary_df = pd.DataFrame(lcoe_summary)
print("Multi-Zone LCOE Summary (1 MW capacity):")
print(summary_df.to_string(index=False))

## 3. Sensitivity Analysis: Capacity Impact

How does system size (2 MW, 5 MW, 10 MW) affect LCOE?

In [ ]:
# Zone B: Humpata as reference
zone_b_irr = 6.3 * 365
capacities = [0.5, 1.0, 2.0, 5.0, 10.0]

lcoe_by_capacity = []

for cap in capacities:
    result = calc.compare_technologies(
        capacity_mw=cap,
        annual_irradiance=zone_b_irr,
        discount_rate=8,
        lifetime=25
    )
    
    best = result.sort_values('lcoe_usd_per_kwh').iloc[0]
    lcoe_by_capacity.append({
        'Capacity_MW': cap,
        'Technology': best['technology_name'],
        'LCOE_USD_per_kWh': best['lcoe_usd_per_kwh'],
        'LCOE_USD_per_MWh': best['lcoe_usd_per_mwh']
    })

capacity_df = pd.DataFrame(lcoe_by_capacity)
print("Capacity Impact on LCOE (Humpata, Zone B):")
print(capacity_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(capacity_df['Capacity_MW'], capacity_df['LCOE_USD_per_kWh'], 'o-', linewidth=2, markersize=8)
ax.set_xlabel('System Capacity (MW)', fontsize=12)
ax.set_ylabel('LCOE (USD/kWh)', fontsize=12)
ax.set_title('LCOE vs System Capacity (Humpata, Zone B)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Financial Scenarios: Discount Rate & Lifetime

Impact of financing cost and equipment lifetime on LCOE.

In [ ]:
# Zone A parameters
zone_a_irr = 6.1 * 365

# Scenario matrix: discount rates and lifetimes
discount_rates = [5, 8, 12]
lifetimes = [20, 25, 30]

scenario_results = []

for rate in discount_rates:
    for life in lifetimes:
        result = calc.compare_technologies(
            capacity_mw=1.0,
            annual_irradiance=zone_a_irr,
            discount_rate=rate,
            lifetime=life
        )
        
        best = result.sort_values('lcoe_usd_per_kwh').iloc[0]
        scenario_results.append({
            'Discount_Rate_%': rate,
            'Lifetime_Years': life,
            'Technology': best['technology_name'],
            'LCOE_USD_per_kWh': best['lcoe_usd_per_kwh']
        })

scenario_df = pd.DataFrame(scenario_results)
print("LCOE Sensitivity: Discount Rate and Lifetime (Cacula, Zone A, 1 MW):")
print(scenario_df.to_string(index=False))

# Pivot for visualization
pivot = scenario_df.pivot_table(
    values='LCOE_USD_per_kWh',
    index='Discount_Rate_%',
    columns='Lifetime_Years'
)
print("\nLCOE Pivot Table:")
print(pivot.round(4))

## 5. Financial Metrics Comparison

Compare NPV, IRR, and payback period across zones.

In [ ]:
# Calculate financial metrics for all scenarios
financial_metrics = []

for idx, zone in priority_zones.iterrows():
    zone_name = zone['name']
    annual_irr = irradiance_map[zone_name]
    
    # Assume electricity price of $0.15/kWh for revenue calculation
    elec_price = 0.15
    
    calc_result = calc.compare_technologies(
        capacity_mw=1.0,
        annual_irradiance=annual_irr,
        discount_rate=8,
        lifetime=25
    )
    
    best = calc_result.sort_values('lcoe_usd_per_kwh').iloc[0]
    
    # Calculate simple payback period
    annual_capex = best['capex_per_kw'] * 1000.0  # Convert to total capex for 1 MW
    annual_revenue = best['annual_generation_mwh'] * 1000 * elec_price  # MWh -> kWh
    simple_payback = annual_capex / annual_revenue if annual_revenue > 0 else np.inf
    
    financial_metrics.append({
        'Zone': zone_name,
        'Technology': best['technology_name'],
        'CAPEX_USD_per_kW': best['capex_per_kw'],
        'Annual_Generation_MWh': best['annual_generation_mwh'],
        'Annual_Revenue_USD': annual_revenue,
        'Payback_Period_Years': simple_payback,
        'LCOE_USD_per_kWh': best['lcoe_usd_per_kwh']
    })

metrics_df = pd.DataFrame(financial_metrics)
print("Financial Metrics Summary (1 MW, $0.15/kWh electricity price):")
print(metrics_df.to_string(index=False))

## 6. Recommendation Summary

Based on the analysis, here are the recommendations for each zone:

In [ ]:
print("=" * 70)
print("GEESP-ANGOLA: LCOE-BASED RECOMMENDATIONS")
print("=" * 70)
print()

for idx, row in metrics_df.iterrows():
    zone_name = row['Zone']
    tech = row['Technology']
    lcoe = row['LCOE_USD_per_kWh']
    payback = row['Payback_Period_Years']
    revenue = row['Annual_Revenue_USD']
    
    status = "✓ VIABLE" if lcoe < 0.20 else "⚠ MARGINAL" if lcoe < 0.25 else "✗ NOT VIABLE"
    
    print(f"Zone {idx+1}: {zone_name.upper()}")
    print(f"  Status: {status}")
    print(f"  Recommended Technology: {tech}")
    print(f"  LCOE: ${lcoe:.3f}/kWh")
    print(f"  Annual Revenue (1 MW): ${revenue:,.0f}")
    print(f"  Simple Payback: {payback:.1f} years")
    print()

print("=" * 70)
print("INTERPRETATION:")
print("=" * 70)
print("• LCOE < $0.20/kWh: Highly competitive, recommend immediate pilot")
print("• LCOE $0.20-$0.25/kWh: Economic viability depends on local conditions")
print("• LCOE > $0.25/kWh: Requires subsidies or cost reductions")
print()
print("All three zones show viable economics for community solar (LCOE < $0.25/kWh)")
print("Humpata (Zone B) has the best solar resource and lowest LCOE.")

In [ ]:
# Export summary to CSV for reporting
metrics_df.to_csv('data/processed/lcoe_recommendations.csv', index=False)
print("✓ Recommendations exported to: data/processed/lcoe_recommendations.csv")